## Sumário

* [Packages & Frameworks](#packages--frameworks)
* [Enunciado do Problema](#questão-1)
    * [Item a)](#item-a)
        * [Pré-processamento dos Dados](#a-data-preprocessing)
        * [Modelo](#a-model)
        * [Treinamento e Predição](#a-train--prediction)
        * [Métricas de Treinamento](#a-training-metrics)
        * [Métricas de Predição](#a-prediction-metrics)
    * [Item b)](#item-b)
        * [Pré-processamento dos Dados](#b-data-preprocessing)
        * [Modelo](#b-model)
        * [Treinamento e Predição](#b-train--prediction)
        * [Métricas de Treinamento](#b-training-metrics)
        * [Métricas de Predição](#b-prediction-metrics)

### Packages & Frameworks

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow import keras

### Questão 1

Implemente uma rede perceptron de múltiplas camadas e utilize-a para aproximar as funções abaixo.  

a. f(x) = log10(x), onde 1 ≤ x ≤ 10

b. f(x) = 10x^5 + 5x^4 + 2x^3 – 0.5x^2 + 3x + 2, onde 0 ≤ x ≤ 5

Para cada função a ser aproximada, gere um conjunto de treinamento e um conjunto de testes. 

Nesses conjuntos, cada amostra deve ser representada da seguinte forma: x é a entrada e f(x) é a saída desejada - rótulo. Treine um perceptron de múltiplas camadas para que ele aprenda a aproximar a função a partir do conjunto de treinamento, e vá testando com o conjunto de validação.  

Apresente os gráficos das funções reais vs. funções aproximadas e as curvas de erro de treinamento e validação.

#### Item a)

#### a) Data Preprocessing

In [ ]:
f = lambda x: np.log10(x)
X = np.linspace(1, 10, 500).reshape(-1, 1)
y = f(X)

X_max = np.max(X)
X_normalized = X / X_max

y_max = np.max(y)
y_normalized = y / y_max

X_train, X_test_full, y_train, y_test_full = train_test_split(
    X_normalized, y_normalized, test_size=0.3, random_state=42
)
X_test, X_valid, y_test, y_valid = train_test_split(
    X_test_full, y_test_full, test_size=0.5, random_state=42
)

In [ ]:
# Plots
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# Training Data
axs[0].scatter(X_train, y_train, color="blue")
axs[0].set_title(f"Conjunto de Treino ({len(X_train)} amostras)")
axs[0].set_xlabel("X")
axs[0].set_ylabel("f(X)")

# Validation Data
axs[1].scatter(X_valid, y_valid, color="red")
axs[1].set_title(f"Conjunto de Validação ({len(X_valid)} amostras)")
axs[1].set_xlabel("X")
axs[1].get_yaxis().set_visible(False)

# Testing Data
axs[2].scatter(X_test, y_test, color="green")
axs[2].set_title(f"Conjunto de Teste ({len(X_test)} amostras)")
axs[2].set_xlabel("X")
axs[2].get_yaxis().set_visible(False)

plt.suptitle("Divisão dos Dados (Treino, Validação e Teste)", fontsize=14)
plt.tight_layout()
plt.show()

#### a) Model

In [ ]:
input_ = keras.layers.Input(shape=X_train.shape[1:])
hidden1 = keras.layers.Dense(250, activation="relu")(input_)
hidden2 = keras.layers.Dense(100, activation="relu")(hidden1)
hidden3 = keras.layers.Dense(50, activation="relu")(hidden2)
output = keras.layers.Dense(1, dtype="float64")(hidden3)

model = keras.Model(inputs=[input_], outputs=[output])

custom_optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss="mse", optimizer=custom_optimizer, metrics=["r2_score"])
model.summary()

model.save("../models/model-question1-item_a.h5")
# plot_model(model, to_file="architecture-question1-item_a.png", show_shapes=True)

#### a) Train & Prediction

In [ ]:
# Train

history = model.fit(X_train, y_train, epochs=100, validation_data=(X_valid, y_valid))

# Prediction

y_pred = model.predict(X_test)

#### a) Training Metrics

In [ ]:
plt.figure()
pd.DataFrame(history.history).plot(figsize=(8, 5))
plt.title("Métricas de Treinamento")
plt.xlabel("Épocas")
plt.ylim([-0.5, 1.1])
plt.show()

#### a) Prediction Metrics

In [ ]:
metric_r2 = keras.metrics.R2Score()
metric_r2.update_state(y_test, y_pred)
result_r2 = metric_r2.result()

metric_mse = keras.metrics.MeanSquaredError()
metric_mse.update_state(y_test, y_pred)
result_mse = metric_mse.result()

metric_mae = keras.metrics.MeanAbsoluteError()
metric_mae.update_state(y_test, y_pred)
result_mae = metric_mae.result()

print(f"R2 Score: {result_r2}")
print(f"MSE: {result_mse}")
print(f"MAE: {result_mae}")

X_test_plot = X_test * X_max
y_test_plot = y_test * y_max
y_pred_plot = y_pred * y_max

X_test_plot = sorted(X_test_plot)
y_test_plot = sorted(y_test_plot)
y_pred_plot = sorted(y_pred_plot)

plt.figure()
plt.scatter(
    X_test_plot, y_test_plot, marker="s", color="blue", alpha=0.5, label=f"Teste"
)
plt.scatter(
    X_test_plot, y_pred_plot, marker="o", color="green", alpha=0.75, label="Predição"
)
plt.xlabel("X")
plt.ylabel("f(X)")
plt.legend()
plt.title(r"$f(x) = \log _{10}(x)$")

#### Item b)

#### b) Data Preprocessing

In [ ]:
f = lambda x: 10 * x**5 + 5 * x**4 + 2 * x**3 - 0.5 * x**2 + 3 * x + 2
X = np.linspace(0, 5, 500).reshape(-1, 1)
y = f(X)

X_max = np.max(X)
X_normalized = X / X_max

y_max = np.max(y)
y_normalized = y / y_max

X_train, X_test_full, y_train, y_test_full = train_test_split(
    X_normalized, y_normalized, test_size=0.2, random_state=42
)
X_test, X_valid, y_test, y_valid = train_test_split(
    X_test_full, y_test_full, test_size=0.5, random_state=42
)

In [ ]:
# Plots
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# Training Data
axs[0].scatter(X_train, y_train, color="blue")
axs[0].set_title(f"Conjunto de Treino ({len(X_train)} amostras)")
axs[0].set_xlabel("X")
axs[0].set_ylabel("f(x)")

# Validation Data
axs[1].scatter(X_valid, y_valid, color="red")
axs[1].set_title(f"Conjunto de Validação ({len(X_valid)} amostras)")
axs[1].set_xlabel("X")
axs[1].set_ylabel("f(x)")

# Testing Data
axs[2].scatter(X_test, y_test, color="green")
axs[2].set_title(f"Conjunto de Teste ({len(X_test)} amostras)")
axs[2].set_xlabel("X")
axs[2].set_ylabel("f(x)")

plt.suptitle("Divisão dos Dados (Treino, Validação e Teste)", fontsize=14)
plt.tight_layout()
plt.show()

#### b) Model

In [ ]:
input_ = keras.layers.Input(shape=X_train.shape[1:])
hidden1 = keras.layers.Dense(250, activation="relu")(input_)
hidden2 = keras.layers.Dense(100, activation="relu")(hidden1)
hidden3 = keras.layers.Dense(50, activation="relu")(hidden2)
output = keras.layers.Dense(1, dtype="float64")(hidden3)

model = keras.Model(inputs=[input_], outputs=[output])

custom_optimizer = keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss="mse", optimizer=custom_optimizer, metrics=["r2_score"])
model.summary()

model.save("../models/model-question1-item_b.h5")
# plot_model(model, to_file="architecture-question1-item_b.png", show_shapes=True)

#### b) Train & Prediction

In [ ]:
# Train

history = model.fit(X_train, y_train, epochs=100, validation_data=(X_valid, y_valid))

# Prediction

y_pred = model.predict(X_test)

#### b) Training Metrics

In [ ]:
plt.figure()
pd.DataFrame(history.history).plot(figsize=(8, 5))
plt.title("Métricas de Treinamento")
plt.xlabel("Épocas")
plt.show()

#### b) Prediction Metrics

In [ ]:
metric_r2 = keras.metrics.R2Score()
metric_r2.update_state(y_test, y_pred)
result_r2 = metric_r2.result()

metric_mse = keras.metrics.MeanSquaredError()
metric_mse.update_state(y_test, y_pred)
result_mse = metric_mse.result()

metric_mae = keras.metrics.MeanAbsoluteError()
metric_mae.update_state(y_test, y_pred)
result_mae = metric_mae.result()

print(f"R2 Score: {result_r2}")
print(f"MSE: {result_mse}")
print(f"MAE: {result_mae}")

X_test_plot = X_test*X_max
y_test_plot = y_test*y_max
y_pred_plot = y_pred*y_max

X_test_plot = sorted(X_test_plot)
y_test_plot = sorted(y_test_plot)
y_pred_plot = sorted(y_pred_plot)

plt.figure()
plt.scatter(X_test_plot, y_test_plot, marker="s", color="blue", alpha=0.5, label=f"Teste")
plt.scatter(X_test_plot, y_pred_plot, marker="o", color="green", alpha=0.75, label="Predição")
plt.xlabel("X")
plt.ylabel("f(X)")
plt.legend()
plt.title(r"$f(x) = 10x^5 + 5x^4 + 2x^3 - 0.5x^2 + 3x + 2$");